# Measured PPO training and checkpoint evaluation

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
"""One fresh training or checkpoint-evaluation process for the Phase1 launcher."""
from pathlib import Path
import argparse,hashlib,json,os,sys,time,traceback,signal
from contract import ROOT,CODE,read_json,write_json,sha

def fingerprint(value):
    h=hashlib.sha256()
    def add(v):
        if hasattr(v,'detach'):v=v.detach().cpu().numpy()
        if hasattr(v,'dtype') and hasattr(v,'tobytes'):
            h.update(str(v.dtype).encode());h.update(str(v.shape).encode());h.update(v.tobytes())
        elif isinstance(v,dict):
            for key in sorted(v,key=str):h.update(str(key).encode());add(v[key])
        elif isinstance(v,(tuple,list)):
            for x in v:add(x)
        else:h.update(repr(v).encode())
    add(value);return h.hexdigest()

def main():
    parser=argparse.ArgumentParser();parser.add_argument('--mode',choices=['train','evaluate'],required=True)
    parser.add_argument('--resolved',required=True);parser.add_argument('--stage-dir',required=True)
    parser.add_argument('--simulator-seed',type=int,required=True);parser.add_argument('--checkpoint');parser.add_argument('--training-dir')
    args=parser.parse_args();stage=Path(args.stage_dir).resolve();stage.mkdir(exist_ok=False)
    cfg=read_json(args.resolved);started=time.perf_counter();result={'mode':args.mode,'status':'RUNNING','pid':os.getpid(),'config_sha256':sha(args.resolved),'simulator_seed':args.simulator_seed}
    os.environ['XDG_CONFIG_HOME']=str(stage/'config-home');os.environ['MPLCONFIGDIR']=str(stage/'config-home/matplotlib')
    os.chdir(CODE);sys.path.insert(0,str(CODE));runtime=None
    def interrupt(signum,frame):raise KeyboardInterrupt(f'signal {signum}')
    signal.signal(signal.SIGTERM,interrupt);signal.signal(signal.SIGINT,interrupt)
    print('RESOLVED_SETTINGS '+json.dumps(cfg,sort_keys=True),flush=True)
    try:
        import numpy as np
        import torch
        torch.set_num_threads(cfg['torch_num_threads']);torch.set_num_interop_threads(cfg['torch_num_interop_threads'])
        from stable_baselines3 import PPO
        from stable_baselines3.common.callbacks import BaseCallback
        from stable_baselines3.common.monitor import Monitor
        from runtime import Runtime
        write_json(stage/'runtime_versions.json',{'python':sys.version,'torch':torch.__version__,'sb3':__import__('stable_baselines3').__version__,
            'numpy':np.__version__,'torch_threads':torch.get_num_threads(),'torch_interop_threads':torch.get_num_interop_threads(),'device':'cpu'})
        runtime=Runtime(cfg,stage,args.simulator_seed)
        updates=[];callback_metrics=[]
        if args.mode=='train':
            env=runtime.create();monitor=Monitor(env,filename=str(stage/'monitor.csv'));runtime.monitor=monitor
            kwargs=dict(cfg['ppo']);policy_kwargs=dict(kwargs['policy_kwargs'])
            policy_kwargs['activation_fn']=torch.nn.Tanh;policy_kwargs['optimizer_class']=torch.optim.Adam
            kwargs['policy_kwargs']=policy_kwargs
            class MeasuredPPO(PPO):
                def train(self):
                    before={k:v.detach().cpu().clone() for k,v in self.policy.state_dict().items()}
                    start=time.perf_counter();super().train();elapsed=time.perf_counter()-start
                    after=self.policy.state_dict()
                    if not all(torch.isfinite(v).all() for v in after.values()):raise ValueError('Nonfinite policy parameters')
                    changes={k:float(torch.max(torch.abs(after[k].cpu()-v))) for k,v in before.items()}
                    checkpoint=None
                    if self.num_timesteps%cfg['checkpoint_interval_decisions']==0 or self.num_timesteps==cfg['training_decisions']:
                        checkpoint=stage/f'checkpoint_{self.num_timesteps}.zip'
                        if checkpoint.exists():raise FileExistsError(checkpoint)
                        self.save(str(checkpoint))
                    row={'num_timesteps':self.num_timesteps,'ppo_epoch_updates':self._n_updates,'optimization_seconds':elapsed,
                        'changed_parameter_tensors':sum(v>0 for v in changes.values()),'max_parameter_abs_change':max(changes.values()),
                        'policy_state_sha256':fingerprint(after),'optimizer_state_sha256':fingerprint(self.policy.optimizer.state_dict()),
                        'checkpoint':None if checkpoint is None else checkpoint.name,'checkpoint_sha256':None if checkpoint is None else sha(checkpoint)}
                    if row['changed_parameter_tensors']==0:raise AssertionError('No parameter update')
                    updates.append(row);write_json(stage/'updates.json',updates);print('PPO_UPDATE '+json.dumps(row),flush=True)
            class Counts(BaseCallback):
                def _on_step(self):
                    if self.num_timesteps!=env.decisions:raise AssertionError('PPO/native decision counts diverged')
                    if self.num_timesteps%256==0:
                        def memory(pid):
                            data=Path(f'/proc/{pid}/status').read_text().splitlines()
                            return {line.split(':')[0]:int(line.split()[1])*1024 for line in data if line.startswith(('VmRSS:','VmHWM:'))}
                        row={'num_timesteps':self.num_timesteps,'wall_seconds':time.perf_counter()-started,
                            'python':memory(os.getpid()),'unity':memory(runtime.process.pid)}
                        callback_metrics.append(row);write_json(stage/'resource_samples.json',callback_metrics)
                    return True
            model=MeasuredPPO(cfg['policy'],monitor,**kwargs)
            initial={k:v.detach().cpu().numpy().copy() for k,v in model.policy.state_dict().items()}
            np.savez(stage/'initial_policy_parameters.npz',**initial)
            initial_hash=fingerprint(model.policy.state_dict());training_start=time.perf_counter()
            model.learn(total_timesteps=cfg['training_decisions'],callback=Counts(),reset_num_timesteps=True,progress_bar=False)
            learning_seconds=time.perf_counter()-training_start
            if model.num_timesteps!=cfg['training_decisions'] or env.decisions!=cfg['training_decisions']:raise AssertionError('Wrong actual decision count')
            if len(updates)!=cfg['rollouts']:raise AssertionError('Wrong rollout/update count')
            final={k:v.detach().cpu().numpy().copy() for k,v in model.policy.state_dict().items()}
            np.savez(stage/'final_policy_parameters.npz',**final)
            final_hash=fingerprint(model.policy.state_dict())
            if initial_hash==final_hash:raise AssertionError('Policy did not change')
            observations=np.asarray(env.samples,dtype=np.float32)
            actions,_=model.predict(observations,deterministic=True)
            np.save(stage/'expected_actions.npy',actions)
            result.update({'actual_training_decisions':model.num_timesteps,'native_decisions':env.decisions,'completed_rollouts':len(updates),
                'ppo_epoch_updates':model._n_updates,'initial_policy_state_sha256':initial_hash,'final_policy_state_sha256':final_hash,
                'final_optimizer_state_sha256':fingerprint(model.policy.optimizer.state_dict()),
                'final_checkpoint':f'checkpoint_{model.num_timesteps}.zip','final_checkpoint_sha256':sha(stage/f'checkpoint_{model.num_timesteps}.zip'),
                'saved_observation_count':len(observations),'expected_actions_sha256':sha(stage/'expected_actions.npy'),
                'learning_wall_seconds':learning_seconds,'optimization_seconds':sum(r['optimization_seconds'] for r in updates),
                'parameters_changed':True,'source_config_sha256':sha(args.resolved)})
        else:
            if not args.checkpoint or not args.training_dir:raise ValueError('Evaluation needs checkpoint and training evidence')
            training=Path(args.training_dir);trained=read_json(training/'result.json');checkpoint=Path(args.checkpoint)
            if sha(checkpoint)!=trained['final_checkpoint_sha256'] or sha(args.resolved)!=trained['source_config_sha256']:
                raise ValueError('Checkpoint or training config hash mismatch')
            model=PPO.load(str(checkpoint),device='cpu')
            if fingerprint(model.policy.state_dict())!=trained['final_policy_state_sha256']:raise AssertionError('Reloaded weights differ')
            if fingerprint(model.policy.optimizer.state_dict())!=trained['final_optimizer_state_sha256']:raise AssertionError('Reloaded optimizer differs')
            samples=np.load(training/'sample_observations.npy',allow_pickle=False)
            expected=np.load(training/'expected_actions.npy',allow_pickle=False)
            actual,_=model.predict(samples,deterministic=True);np.testing.assert_array_equal(actual,expected)
            np.save(stage/'reloaded_actions.npy',actual)
            env=runtime.create();obs,reset_info=env.reset()
            for count in range(1,601):
                action,_=model.predict(obs,deterministic=True)
                obs,reward,terminated,truncated,info=env.step(action)
                if terminated or truncated:break
            else:raise AssertionError('Evaluation did not end within horizon')
            if fingerprint(model.policy.state_dict())!=trained['final_policy_state_sha256']:raise AssertionError('Evaluation altered weights')
            result.update({'checkpoint_sha256':sha(checkpoint),'reloaded_policy_state_sha256':fingerprint(model.policy.state_dict()),
                'reloaded_optimizer_state_sha256':fingerprint(model.policy.optimizer.state_dict()),'saved_observation_actions_match':True,
                'saved_observation_count':len(samples),'evaluation_decisions':count,'terminated':terminated,'truncated':truncated,
                'episode_end_reason':info['episode_end_reason'],'raw_score':info['raw_score_signal'],'PPO_learning_decisions':0})
        result['status']='PASS'
    except BaseException as exc:
        result['status']='FAIL';result['exception']=repr(exc);traceback.print_exc()
    finally:
        if runtime is not None:
            try:result['runtime']=runtime.close()
            except BaseException as exc:result['status']='FAIL';result['cleanup_or_preservation_error']=repr(exc);traceback.print_exc()
        result['wall_seconds']=time.perf_counter()-started
        write_json(stage/'result.json',result);print('STAGE_RESULT '+json.dumps(result),flush=True)
    return 0 if result['status']=='PASS' else 1
print('Measured PPO training and checkpoint evaluation definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Measured PPO training and checkpoint evaluation definitions/execution completed.
